# 🧹 Clase 4 — Limpieza y preparación de datos

Los datos del mundo real están **sucios**: valores faltantes, errores, outliers, formatos inconsistentes.
Antes de entrenar un modelo, hay que limpiar. En esta clase aprendés cómo.

### ¿Qué vamos a cubrir?

| Sección | Concepto |
|---|---|
| 1 | Setup y generar dataset con problemas |
| 2 | Diagnóstico: ¿qué tan sucio está? |
| 3 | Valores faltantes (NaN): detección y estrategias |
| 4 | Outliers y errores de rango |
| 5 | Datos duplicados |
| 6 | Visualización antes vs después |
| 7 | Pipeline de limpieza reutilizable |
| 8 | Ejercicio evaluable |

> **Garbage in, garbage out.** Si entrenás un modelo con datos sucios, las predicciones van a ser basura.

In [ ]:
# ── 1. Setup ────────────────────────────────────────────────

import pandas as pd
import matplotlib.pyplot as plt
import random
import math

random.seed(42)

print(f"✅ pandas     : {pd.__version__}")
print(f"✅ matplotlib : {plt.matplotlib.__version__}")

In [ ]:
# Generar dataset CON PROBLEMAS deliberados para practicar limpieza

random.seed(42)

nombres_pool = [
    "Ana", "Lucía", "Martín", "Sofía", "Mateo", "Valentina", "Santiago",
    "Isabella", "Sebastián", "Camila", "Nicolás", "Emilia", "Daniel",
    "Renata", "Tomás", "Paula", "Alejandro", "Florencia", "Lucas",
]
apellidos_pool = [
    "García", "Martínez", "López", "González", "Rodríguez", "Pérez",
    "Sánchez", "Ramírez", "Torres", "Flores",
]
carreras = ["Ingeniería", "Sistemas", "Diseño", "Economía"]

N = 200
datos = []

for i in range(N):
    nombre = f"{random.choice(nombres_pool)} {random.choice(apellidos_pool)}"
    edad = random.randint(18, 35)
    carrera = random.choice(carreras)
    trabaja = random.random() < 0.4
    
    base_horas = random.gauss(12, 5) if not trabaja else random.gauss(7, 4)
    horas_estudio = round(max(0, min(30, base_horas)), 1)
    asistencia = round(min(100, max(0, horas_estudio * 4.5 + random.gauss(20, 15))), 1)
    nota_parcial = round(min(10, max(0, 
        horas_estudio * 0.3 + asistencia * 0.03 + random.gauss(1, 1.5))), 1)
    nota_final = round(min(10, max(0,
        nota_parcial * 0.4 + horas_estudio * 0.2 + asistencia * 0.02 + random.gauss(0.5, 1))), 1)
    
    # === INTRODUCIR PROBLEMAS A PROPÓSITO ===
    
    # Problema 1: NaN aleatorios (~10% en varias columnas)
    if random.random() < 0.10:
        nota_parcial = None
    if random.random() < 0.08:
        asistencia = None
    if random.random() < 0.05:
        horas_estudio = None
    
    # Problema 2: outliers / errores de rango
    if random.random() < 0.02:
        edad = 150               # error de tipeo
    if random.random() < 0.02:
        horas_estudio = -5.0     # valor negativo imposible
    if random.random() < 0.02:
        asistencia = 250.0       # porcentaje > 100 = error
    if random.random() < 0.02:
        nota_final = 15.0        # nota > 10 = error
    
    datos.append({
        "nombre": nombre,
        "edad": edad,
        "carrera": carrera,
        "horas_estudio": horas_estudio,
        "nota_parcial": nota_parcial,
        "asistencia_pct": asistencia,
        "trabaja": trabaja,
        "nota_final": nota_final,
    })

# Problema 3: filas duplicadas
for _ in range(5):
    datos.append(datos[random.randint(0, N-1)].copy())

df = pd.DataFrame(datos)

print(f"✅ Dataset con problemas generado: {len(df)} filas, {len(df.columns)} columnas")
print("\n⚠️  Este dataset tiene errores deliberados. ¡Vamos a limpiarlos!")

---
## 2. Diagnóstico: ¿qué tan sucio está?

Antes de limpiar, necesitamos saber **exactamente** qué problemas tiene el dataset.

In [ ]:
# Reporte de salud del dataset

print("═" * 60)
print("  🏥 REPORTE DE SALUD DEL DATASET")
print("═" * 60)

# 1. Dimensiones
print(f"\n📏 Dimensiones: {df.shape[0]} filas × {df.shape[1]} columnas")

# 2. Valores faltantes
print("\n📊 Valores faltantes:")
nulos = df.isnull().sum()
for col in df.columns:
    n = nulos[col]
    if n > 0:
        pct = n / len(df) * 100
        print(f"  ⚠️  {col:<20} {n:>3} NaN ({pct:.1f}%)")

total_nan = nulos.sum()
total_celdas = df.shape[0] * df.shape[1]
print(f"\n  Total: {total_nan} NaN de {total_celdas} celdas ({total_nan/total_celdas*100:.1f}%)")

# 3. Duplicados
dupes = df.duplicated().sum()
print(f"\n🔁 Filas duplicadas: {dupes}")

# 4. Valores fuera de rango
print("\n🔍 Valores fuera de rango esperado:")
rangos = {
    "edad":           (18, 60),
    "horas_estudio":  (0, 40),
    "nota_parcial":   (0, 10),
    "asistencia_pct": (0, 100),
    "nota_final":     (0, 10),
}

for col, (rmin, rmax) in rangos.items():
    serie = df[col].dropna()
    fuera = ((serie < rmin) | (serie > rmax)).sum()
    if fuera > 0:
        print(f"  ⚠️  {col:<20} {fuera:>3} valores fuera de [{rmin}, {rmax}]")

print(f"\n{'═' * 60}")
print("📌 Diagnóstico completo. Ahora a limpiar.")

---
## 3. Valores faltantes (NaN): detección y estrategias

Los NaN (Not a Number) son valores que faltan en el dataset. Todos los modelos de ML necesitan datos completos,
así que tenemos que decidir qué hacer con ellos.

### Estrategias para manejar NaN

| Estrategia | Cuándo usar | Comando pandas |
|---|---|---|
| **Eliminar filas** | Pocos NaN, muchos datos | `df.dropna()` |
| **Rellenar con media** | Variable numérica simétrica | `df.fillna(df.mean())` |
| **Rellenar con mediana** | Variable con outliers | `df.fillna(df.median())` |
| **Rellenar con moda** | Variable categórica | `df.fillna(df.mode()[0])` |
| **Rellenar con 0** | Cuando 0 tiene sentido | `df.fillna(0)` |

> 💡 **Regla de pulgar:** si una columna tiene >30% NaN, considerá eliminar la columna entera.

In [ ]:
# Guardar una copia antes de limpiar (para comparar después)
df_original = df.copy()

# Ver filas con NaN
filas_con_nan = df[df.isnull().any(axis=1)]
print(f"📊 Filas que tienen al menos 1 NaN: {len(filas_con_nan)} de {len(df)}")
print("\nEjemplo de filas con NaN:")
filas_con_nan.head(5)

In [ ]:
# Estrategia: rellenar con la MEDIANA de cada columna numérica
# Usamos mediana porque es más robusta que la media ante outliers

cols_numericas = ["horas_estudio", "nota_parcial", "asistencia_pct"]

print("🔧 Rellenando NaN con la mediana de cada columna:\n")

for col in cols_numericas:
    n_nulos_antes = df[col].isnull().sum()
    if n_nulos_antes > 0:
        mediana = df[col].median()
        df[col] = df[col].fillna(mediana)
        print(f"  ✅ {col}: {n_nulos_antes} NaN → rellenados con mediana = {mediana:.1f}")

# Verificar que no queden NaN
nan_restantes = df.isnull().sum().sum()
print(f"\n📊 NaN restantes en todo el dataset: {nan_restantes}")

In [ ]:
# Comparemos: ¿cómo cambiaron las estadísticas después de rellenar?

print("📊 Comparación ANTES vs DESPUÉS de rellenar NaN\n")

for col in cols_numericas:
    antes_media = df_original[col].mean()
    antes_std   = df_original[col].std()
    despues_media = df[col].mean()
    despues_std   = df[col].std()
    
    print(f"{col}:")
    print(f"  Antes:   media={antes_media:.2f}, std={antes_std:.2f}")
    print(f"  Después: media={despues_media:.2f}, std={despues_std:.2f}")
    cambio = abs(despues_media - antes_media) / antes_media * 100 if antes_media else 0
    print(f"  Cambio en la media: {cambio:.1f}%")
    print()

print("💡 Rellenar con mediana cambia muy poco las estadísticas → es una buena estrategia.")

---
## 4. Outliers y errores de rango

Los outliers son valores extremos que pueden ser:
- **Errores de datos** (edad = 150) → hay que corregir o eliminar
- **Valores reales extremos** (un CEO gana 100x el promedio) → depende del contexto

### Métodos de detección

| Método | Ventaja | Cuándo usar |
|---|---|---|
| **Rango fijo** | Simple, intuitivo | Cuando conocés los límites lógicos |
| **IQR** | Estadístico, automático | Cuando no conocés los límites |
| **Z-score** | Basado en desviación estándar | Datos normalmente distribuidos |

In [ ]:
# Método 1: Detección por rango fijo
# Sabemos los límites lógicos de cada variable

rangos = {
    "edad":           (18, 60),
    "horas_estudio":  (0, 40),
    "nota_parcial":   (0, 10),
    "asistencia_pct": (0, 100),
    "nota_final":     (0, 10),
}

print("🔍 Outliers detectados por rango fijo:\n")

outliers_totales = 0
for col, (rmin, rmax) in rangos.items():
    mask = (df[col] < rmin) | (df[col] > rmax)
    n_outliers = mask.sum()
    if n_outliers > 0:
        valores = df.loc[mask, col].tolist()
        print(f"  ⚠️  {col}: {n_outliers} outliers → {valores}")
        outliers_totales += n_outliers

print(f"\n📊 Total de outliers por rango fijo: {outliers_totales}")

In [ ]:
# Método 2: Detección IQR (Interquartile Range)
# Outlier = cualquier valor fuera de [Q1 - 1.5*IQR, Q3 + 1.5*IQR]

print("🔍 Outliers detectados por IQR (método estadístico):\n")

for col in ["horas_estudio", "nota_parcial", "asistencia_pct", "nota_final"]:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lim_inf = Q1 - 1.5 * IQR
    lim_sup = Q3 + 1.5 * IQR
    
    mask_iqr = (df[col] < lim_inf) | (df[col] > lim_sup)
    n_out = mask_iqr.sum()
    
    print(f"  {col}:")
    print(f"    Q1={Q1:.1f}, Q3={Q3:.1f}, IQR={IQR:.1f}")
    print(f"    Límites: [{lim_inf:.1f}, {lim_sup:.1f}]")
    print(f"    Outliers: {n_out}")
    print()

In [ ]:
# Corregir outliers: reemplazar con NaN y luego rellenar

print("🔧 Corrigiendo valores fuera de rango:\n")

filas_antes = len(df)

for col, (rmin, rmax) in rangos.items():
    mask = (df[col] < rmin) | (df[col] > rmax)
    n_corregidos = mask.sum()
    if n_corregidos > 0:
        mediana = df.loc[~mask, col].median()  # mediana sin outliers
        df.loc[mask, col] = mediana
        print(f"  ✅ {col}: {n_corregidos} valores → reemplazados con mediana = {mediana:.1f}")

# Verificar
print("\n📊 Verificación de rangos después de limpieza:")
for col, (rmin, rmax) in rangos.items():
    vmin, vmax = df[col].min(), df[col].max()
    ok = vmin >= rmin and vmax <= rmax
    emoji = "✅" if ok else "❌"
    print(f"  {emoji} {col}: [{vmin:.1f}, {vmax:.1f}] (esperado: [{rmin}, {rmax}])")

---
## 5. Datos duplicados

Las filas duplicadas inflan artificialmente el dataset y pueden sesgar el modelo.

In [ ]:
# Detectar y eliminar duplicados

dupes = df.duplicated()
n_dupes = dupes.sum()

print(f"🔁 Filas duplicadas encontradas: {n_dupes}")

if n_dupes > 0:
    print("\nEjemplo de duplicados:")
    print(df[dupes].head())
    
    # Eliminar duplicados
    df = df.drop_duplicates()
    print(f"\n✅ Duplicados eliminados. Filas restantes: {len(df)}")
else:
    print("✅ No hay duplicados.")

---
## 6. Visualización: antes vs después

Veamos cómo cambió el dataset después de toda la limpieza.

In [ ]:
# Comparación visual antes vs después

cols_comparar = ["horas_estudio", "nota_parcial", "asistencia_pct", "nota_final"]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))

for i, col in enumerate(cols_comparar):
    # Antes
    ax_antes = axes[0][i]
    data_antes = df_original[col].dropna()
    ax_antes.hist(data_antes, bins=20, color="#ff6b6b", alpha=0.7, edgecolor="#333")
    ax_antes.set_title(f"ANTES — {col}", fontsize=10, fontweight="bold")
    ax_antes.grid(True, alpha=0.2)
    
    # Después
    ax_despues = axes[1][i]
    data_despues = df[col].dropna()
    ax_despues.hist(data_despues, bins=20, color="#6bcb77", alpha=0.7, edgecolor="#333")
    ax_despues.set_title(f"DESPUÉS — {col}", fontsize=10, fontweight="bold")
    ax_despues.grid(True, alpha=0.2)

fig.suptitle("Distribuciones ANTES vs DESPUÉS de la limpieza",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print("📊 Rojo = antes (con errores) | Verde = después (limpio)")
print("   Los outliers extremos desaparecieron y las distribuciones son más coherentes.")

In [ ]:
# Resumen numérico de la limpieza

print("═" * 60)
print("  📋 RESUMEN DE LIMPIEZA")
print("═" * 60)
print(f"\n  Filas originales:    {len(df_original)}")
print(f"  Filas después:       {len(df)}")
print(f"  Filas eliminadas:    {len(df_original) - len(df)} (duplicados)")

print(f"\n  NaN originales:      {df_original.isnull().sum().sum()}")
print(f"  NaN después:         {df.isnull().sum().sum()}")

print("\n  Correcciones realizadas:")
print("    ✅ NaN → rellenados con mediana")
print("    ✅ Outliers de rango → reemplazados con mediana")
print("    ✅ Duplicados → eliminados")
print(f"\n{'═' * 60}")

---
## 7. Pipeline de limpieza reutilizable

En la práctica querés **automatizar** la limpieza en una función reutilizable.

In [ ]:
def limpiar_dataset(df, rangos_validos=None, verbose=True):
    """
    Pipeline de limpieza estándar.
    
    Pasos:
    1. Eliminar duplicados
    2. Corregir valores fuera de rango → NaN
    3. Rellenar NaN con mediana (numéricos)
    
    Parámetros:
        df: DataFrame a limpiar
        rangos_validos: dict {columna: (min, max)}
        verbose: si True, imprime el reporte
    
    Retorna: DataFrame limpio
    """
    df_limpio = df.copy()
    reporte = {}
    
    # Paso 1: duplicados
    n_dupes = df_limpio.duplicated().sum()
    df_limpio = df_limpio.drop_duplicates()
    reporte["duplicados_eliminados"] = n_dupes
    
    # Paso 2: valores fuera de rango → NaN
    outliers_corregidos = 0
    if rangos_validos:
        for col, (rmin, rmax) in rangos_validos.items():
            if col in df_limpio.columns:
                mask = (df_limpio[col] < rmin) | (df_limpio[col] > rmax)
                n = mask.sum()
                if n > 0:
                    df_limpio.loc[mask, col] = None  # convertir a NaN
                    outliers_corregidos += n
    reporte["outliers_corregidos"] = outliers_corregidos
    
    # Paso 3: rellenar NaN con mediana
    nan_antes = df_limpio.isnull().sum().sum()
    cols_num = df_limpio.select_dtypes(include=["int64", "float64"]).columns
    for col in cols_num:
        mediana = df_limpio[col].median()
        df_limpio[col] = df_limpio[col].fillna(mediana)
    reporte["nan_rellenados"] = nan_antes
    
    if verbose:
        print(f"✅ Limpieza completada:")
        print(f"   • {reporte['duplicados_eliminados']} duplicados eliminados")
        print(f"   • {reporte['outliers_corregidos']} outliers corregidos")
        print(f"   • {reporte['nan_rellenados']} NaN rellenados")
        print(f"   • Resultado: {len(df_limpio)} filas × {len(df_limpio.columns)} columnas")
    
    return df_limpio

# Demostración: limpiar el dataset original de cero
df_limpio = limpiar_dataset(
    df_original,
    rangos_validos={
        "edad":           (18, 60),
        "horas_estudio":  (0, 40),
        "nota_parcial":   (0, 10),
        "asistencia_pct": (0, 100),
        "nota_final":     (0, 10),
    }
)

In [ ]:
# Verificación final

print("📊 Estado final del dataset limpio:\n")
print(f"  Filas:  {len(df_limpio)}")
print(f"  NaN:    {df_limpio.isnull().sum().sum()}")
print(f"  Dupes:  {df_limpio.duplicated().sum()}")
print()
df_limpio.describe().round(2)

---
## 8. Ejercicio evaluable

### Consigna

Practicá la limpieza de datos respondiendo las preguntas y completando el código.

### Criterio de aprobación

- ✅ Respondiste correctamente las preguntas del diagnóstico
- ✅ Implementaste tu propia estrategia de limpieza
- ✅ El dataset final no tiene NaN, outliers ni duplicados
- ✅ Escribiste la reflexión

In [ ]:
# ══════════════════════════════════════════════════════════════
#  EJERCICIO EVALUABLE — Limpieza de datos
# ══════════════════════════════════════════════════════════════

# ── Parte 1: diagnóstico ────────────────────────────────────

diagnostico = {
    "total_nan_original":        0,   # TODO: ¿cuántos NaN tiene df_original?
    "columna_mas_nan":           "",  # TODO: ¿qué columna tiene más NaN?
    "total_outliers_rango":      0,   # TODO: ¿cuántos valores fuera de rango?
    "total_duplicados":          0,   # TODO: ¿cuántas filas duplicadas?
    "estrategia_nan":            "",  # TODO: ¿qué estrategia elegiste para NaN? (media/mediana/eliminar)
    "por_que_esa_estrategia":    "",  # TODO: ¿por qué elegiste esa estrategia?
}

# Validación
checks = []
checks.append(("total_nan", diagnostico["total_nan_original"] == df_original.isnull().sum().sum()))
checks.append(("duplicados", diagnostico["total_duplicados"] == df_original.duplicated().sum()))
checks.append(("estrategia", diagnostico["estrategia_nan"].strip() != ""))
checks.append(("justificacion", diagnostico["por_que_esa_estrategia"].strip() != ""))

aciertos = sum(ok for _, ok in checks)
print(f"📋 Diagnóstico: {aciertos}/{len(checks)} correctas")
for nombre, ok in checks:
    print(f"  {'✅' if ok else '❌'} {nombre}")

In [ ]:
# ── Parte 2: tu pipeline de limpieza ────────────────────────
#
# Implementá tu propia versión del pipeline de limpieza.
# Partí de df_original y dejalo completamente limpio.
#
# TODO: escribí tu código

mi_df_limpio = df_original.copy()

# Paso 1: eliminar duplicados
# TODO

# Paso 2: corregir outliers
# TODO

# Paso 3: rellenar NaN  
# TODO

# Verificación
print(f"📊 Mi dataset limpio:")
print(f"  Filas: {len(mi_df_limpio)}")
print(f"  NaN:   {mi_df_limpio.isnull().sum().sum()}")
print(f"  Dupes: {mi_df_limpio.duplicated().sum()}")

In [ ]:
# ── Parte 3: reflexión ──────────────────────────────────────

mi_reflexion = ""
# TODO: escribí 3-4 oraciones respondiendo:
# 1. ¿Qué tipo de problema fue más difícil de manejar (NaN, outliers o duplicados)?
# 2. ¿Por qué es importante limpiar datos antes de modelar?
# 3. ¿Qué otro problema podrían tener los datos que no vimos en esta clase?

if mi_reflexion.strip():
    print("✅ Reflexión registrada.")
    print(f"\n{mi_reflexion}")
    print("\n🎉 Guardá este notebook y envialo como entrega de la Clase 4.")
else:
    print("❌ Falta la reflexión. Escribí tu respuesta arriba.")

---
## Resumen de la clase

| Concepto | Lo que aprendiste |
|---|---|
| NaN | Valores faltantes → rellenar con media/mediana/moda o eliminar |
| Outliers | Detección por rango fijo o IQR → reemplazar o eliminar |
| Duplicados | Filas repetidas → `drop_duplicates()` |
| Pipeline | Función reutilizable que automatiza la limpieza |
| Visualización | Comparar distribuciones antes vs después |

### Próxima clase

En la **Clase 5** vamos a transformar los datos limpios en **features** que un modelo puede usar: encoding de categorías, normalización, y partición train/validation/test.

> 📌 **Recordá:** un pipeline de limpieza debe ser reproducible. Si no podés repetirlo, no es ciencia.